In [1]:
# 1 - Importação de bibliotecas

import pandas as pd
from pbixray import PBIXRay

In [2]:
# 2 - Modelo

model = PBIXRay("Novo Cenario Geral.pbix")

In [3]:
# 3 - Medidas DAX em DataFrame

model.dax_measures

,TableName,Name,Expression,DisplayFolder,Description
0,Medidas,Faturamento,"\nSUMX(\n fVendas,\n fVendas[vr_liquido_...",1 - Vendas\Faturamento,None
1,Medidas,Qtde,"\nSUMX(\n fVendas,\n fVendas[qtd_liquida...",1 - Vendas\Qtde,None
2,Medidas,Custo,"\nCALCULATE(\n SUMX(\n fVendas,\n ...",1 - Vendas\Custo,None
3,Medidas,Margem Bruta,[Faturamento] - [Custo],1 - Vendas\Margem Bruta,None
4,Medidas,Faturamento LY,"\nCALCULATE(\n [Faturamento],\n SAMEPERI...",1 - Vendas\Faturamento,None
...,...,...,...,...,...
246,Medidas,Markup Venda Geral,"\nDIVIDE( [Lucro Liquido Geral], [Faturamento] )",4 - Markup\Vendas,None
247,Medidas,Margem Liquida Geral,\n[Faturamento] * [Markup Venda Geral],4 - Markup\Vendas,None
248,Medidas,Lucro Liquido Entrega Geral,\n[Faturamento_Entregas]\n- [Custo_Entregas]\n...,Entregas,None
249,Medidas,Rateio de despesas/Custos Entrega Geral,"\nCALCULATE(\n SUMX(\n fEntregas,\n ...",Entregas,None


In [4]:
# 4 - Código M de cada tabela

model.power_query

,TableName,Expression
0,fVendas,let\n Fonte = PowerPlatform.Dataflows(null)...
1,fDevolucoes,let\n Fonte = PowerPlatform.Dataflows(null)...
2,fVendas_Canceladas,let\n Fonte = PowerPlatform.Dataflows(null)...
3,dProdutos,"let\n Fonte = PowerBI.Dataflows(null),\n ..."
4,dClientes,let\n Fonte = PowerPlatform.Dataflows(null)...
5,dVendedores,let\n Fonte = PowerPlatform.Dataflows(null)...
6,fRecebidos,let\n Fonte = PowerPlatform.Dataflows(null)...
7,fPagos,"let\n Fonte = PowerBI.Dataflows(null),\n ..."
8,dPlano_Contas,"let\n Fonte = PowerBI.Dataflows(null),\n ..."
9,dGrupo_Centro_Custos,"let\n Fonte = PowerBI.Dataflows(null),\n ..."


In [5]:
# 5 - Tabelas do modelo: um DataFrame por tabela

# Lista de tabelas do modelo
todas_tabelas = list(model.tables)

# Tabelas calculadas em DAX não têm dados armazenados — get_table falha nelas
tabelas_calculadas = (
    set(model.dax_tables["TableName"]) if len(model.dax_tables) else set()
)


def carregar_tabelas(nomes=None, ignorar_calculadas=True):
    """Carrega cada tabela do modelo em um DataFrame separado.
    Retorna (dicionário de DataFrames, dicionário de erros)."""
    alvo = nomes or todas_tabelas
    dfs, erros = {}, {}

    for nome in alvo:
        if ignorar_calculadas and nome in tabelas_calculadas:
            continue
        try:
            dfs[nome] = model.get_table(nome)
        except Exception as e:
            erros[nome] = str(e)

    return dfs, erros


# Carga
tabelas, falhas = carregar_tabelas()

# Resumo
resumo = pd.DataFrame(
    [
        {"Tabela": nome, "Linhas": df.shape[0], "Colunas": df.shape[1]}
        for nome, df in tabelas.items()
    ]
).sort_values("Linhas", ascending=False)

print(f"{len(tabelas)} tabelas carregadas\n")
print(resumo.to_string(index=False))
if falhas:
    print("\nNão carregadas:", list(falhas))

20 tabelas carregadas

              Tabela  Linhas  Colunas
             fVendas  825043       53
          fRecebidos  491228       66
         dNum_Vendas  372761        1
            fEstoque  270960        9
fNotas_Fiscais_Saida  129115       35
         fOperadoras  118772       11
           fEntregas  113783       48
              fPagos   38918       23
           dProdutos   18276       39
  fVendas_Canceladas   11263       16
         dCalendario    8035       31
         fDevolucoes    6161       18
          fA_Receber    2911       24
           dClientes    2119       25
            fA_Pagar     934       23
         dFornecedor     725       23
       dPlano_Contas     140       13
         dVendedores      79       24
dGrupo_Centro_Custos      46        3
         dForma_Pgto      10        4


In [6]:
# 6 - Colunas calculadas (DAX) por tabela

def colunas_calculadas(tabela=None):
    dax_cols = model.dax_columns
    if tabela:
        return dax_cols[dax_cols["TableName"] == tabela][["ColumnName", "Expression"]]
    return dax_cols[["TableName", "ColumnName", "Expression"]]


# Uso:
colunas_calculadas("fVendas")   # só a fVendas
# colunas_calculadas()            # todas as tabelas

,ColumnName,Expression
120,comissao_prazo,\nVAR vVendedor = fVendas[cod_vendedor]\nVAR v...
121,rateio,[Rateio de despesas/Custos]
122,taxa_cartao,"\nIF(\n fVendas[TemPagto3] = 1,\n [Taxa ..."
135,TemPagto3,\nVAR vCodVenda = fVendas[cod_venda]\nRETURN\n...
136,numero_nfe,\nVAR vVenda = fVendas[cod_venda]\nVAR vProd ...
137,pis_aliq_Nfe,\nVAR vVenda = fVendas[cod_venda]\nVAR vProd ...
138,cofins_aliq_Nfe,\nVAR vVenda = fVendas[cod_venda]\nVAR vProd ...
139,pis_cofins_nfe,fVendas[pis_aliq_Nfe] + fVendas[cofins_aliq_Nfe]
140,icms_aliq_Nfe,\nVAR vVenda = fVendas[cod_venda]\nVAR vProd ...
141,pis_cofins_aliq,fVendas[pis_aliq] + fVendas[cofins_aliq]
